# 🎬 Test đếm trên VIDEO THẬT — bộ query suite phong phú + lưu video output

Mô phỏng bảng test Excel: **rất nhiều trường hợp query** phân nhóm (cơ bản · màu ·
phụ kiện · hành động · khó/phủ định · **tiếng Việt**) cho từng bài toán. Người tách
2 bài (cắt VẠCH vào/ra + đếm VÙNG). Có **lưu video output** vẽ vạch/vùng + box + số đếm.


## 1) Tải code + cài thư viện


In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
!pip install -q ultralytics 'supervision>=0.21' opencv-python-headless


## 2) Xem catalog + toàn bộ query suite (nhiều trường hợp)


In [ ]:
!python run_scenarios.py --list
import recognition.video_catalog as vc
for t,groups in vc.QUERY_SUITES.items():
    print(f'\n=== {t.upper()} — {sum(len(v) for v in groups.values())} query ===')
    for g,qs in groups.items(): print(f'  [{g}] ' + ' | '.join(qs))


## 3) 🚗 Đếm XE (YOLO) + lưu video output


In [ ]:
!python run_scenarios.py --task vehicles --max-frames 300 --save-dir /kaggle/working/scen_out


## 4) 🚶 Đếm NGƯỜI — cắt VẠCH (vào/ra) + đếm VÙNG + lưu video


In [ ]:
!python run_scenarios.py --task people --max-frames 300 --save-dir /kaggle/working/scen_out


## 5) 📦 Đếm chai trên chuyền (YOLO, nhanh) + lưu video


In [ ]:
!python run_scenarios.py --task conveyor --only milk --max-frames 300 --save-dir /kaggle/working/scen_out


## 6) 🧠 BỘ QUERY SUITE đầy đủ (open-vocab) — NHIỀU TRƯỜNG HỢP như Excel
Chạy toàn bộ query theo nhóm trên 1 video (scorecard có cột **nhóm**). Dùng
LocateAnything (auto cài transformers 4.57.1 + attn sdpa). **Chạy chậm** — giảm
`--max-frames` cho nhanh, dùng `--only` để chọn 1 video.


In [ ]:
# Suite SẢN PHẨM trên video chai (chai/hộp/trạng thái/tiếng Việt…):
!python run_scenarios.py --task conveyor --only milk --suite --max-frames 50 --save-dir /kaggle/working/scen_out


In [ ]:
# Suite NGƯỜI trên video đi bộ (màu/phụ kiện/hành động/khó/tiếng Việt…):
# !python run_scenarios.py --task people --only walk --suite --max-frames 50 --save-dir /kaggle/working/scen_out


In [ ]:
# Suite XE trên video cao tốc:
# !python run_scenarios.py --task vehicles --only 'cao tốc top' --suite --max-frames 50 --save-dir /kaggle/working/scen_out


## 7) 🎥 Xem / tải video output


In [ ]:
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('/kaggle/working/scen_out/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
if vids:
    src = vids[0]; dst = '/kaggle/working/preview_h264.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    print('Xem:', src); display(Video(dst, embed=True, width=700))


---
### Đọc kết quả
- `--suite`: scorecard có cột **nhóm** → so kết quả giữa các nhóm query (dễ/màu/khó/tiếng Việt).
- **det/frame=0** ở query khó / tiếng Việt = model KHÔNG hiểu mô tả đó (kết quả cũng là dữ liệu).
- Tự ra đề: `--queries "a red car,a taxi,xe cứu thương"`. Thêm nhóm query: sửa `QUERY_SUITES` trong `recognition/video_catalog.py`.
